# Training CTLearn models
For this notebook, one would need to download the HDF5 data files
from the CTAO opendata folder (https://cloud.iaa.es/index.php/s/77d6MK4rGfanSKx).


### Check paths, environments, and CTAO test data

In [ ]:
! pwd

In [ ]:
! conda list | grep ctapipe
! conda list | grep ctlearn
! conda list | grep dl1-data-handler 
! conda list | grep keras
! conda list | grep tensorflow

In [ ]:
TRAIN_DIR = "../testdata/ctao_opendata/train"
TEST_DIR = "../testdata/ctao_opendata/test"
! du -h {TRAIN_DIR}/*
! du -h {TEST_DIR}/*

### Explore the CTLearn training tool
Installing CTLearn provides several command-line tools for running and managing different parts of the workflow.
For model training, we use the **ctlearn-train-model** command.
The *-h* option displays a short version of the help message, showing the main command-line options and their usage.
For a complete description of all available options, including the configuration parameters of the different components, use *--help-all*.
Although the command is executed from this Jupyter notebook for convenience and documentation, **ctlearn-train-model** is a command-line tool.
In a typical workflow, the training command would be executed directly from a terminal or submitted as part of a cluster job, for example through sbatch on an HPC system.

In [ ]:
! ctlearn-train-model -h
#! ctlearn-train-model --help-all

## Basic monoscopic CNN-based model training
We first train a basic monoscopic CNN-based model using the default CTLearn model configuration.
The model uses a **ThinResNet** backbone for feature extraction and a standard MLP head for the task-specific prediction.
We consider three different reconstruction tasks, each trained separately:

### 1. Primary particle energy regression
The model reconstructs the energy of the primary particle as a continuous value.
Internally, CTLearn performs the regression on the energy in TeV and on a logarithmic scale, since the primary-particle energy spans a very wide range.
Working in logarithmic space has been found to be more efficient for the regression and is therefore a fixed, non-configurable aspect of the energy reconstruction.
During inference, the prediction tool converts the predicted values back from logarithmic space to the original linear energy scale in TeV.

In [ ]:
! mkdir ../my_outputs
! ctlearn-train-model \
    --signal {TRAIN_DIR} \
    --pattern-signal "gamma-diffuse_with_images_*.dl2.h5" \
    --output "../my_outputs/my_first_training_energy2" \
    --config "../configs/dl1dh_example_config.json" \
    --config "../configs/ctlearn_train_model_example_config.json" \
    --reco energy \
    --verbose

### 2. Arrival direction regression
The model reconstructs the arrival direction of the primary particle.
This can be done in two ways: either using the **cameradirection** task, which transforms the particle origin into the camera frame and performs the regression in camera coordinates, with distances expressed in meters; or using the **skydirection** task, which performs the regression in the sky frame, with field-of-view offsets expressed in degrees.
For monoscopic CNN-based models, we generally recommend using the **cameradirection** task.
During inference, the prediction tool converts the reconstructed camera coordinates back to the original Alt/Az representation, with the final direction expressed in degrees.

In [ ]:
! mkdir ../my_outputs
! ctlearn-train-model \
    --signal {TRAIN_DIR} \
    --pattern-signal "gamma-diffuse_with_images_*.dl2.h5" \
    --output "../my_outputs/my_first_training_cameradirection" \
    --config "../configs/dl1dh_example_config.json" \
    --config "../configs/ctlearn_train_model_example_config.json" \
    --reco cameradirection \
    --TrainCTLearnModel.n_epochs 5 \
    --TrainCTLearnModel.batch_size 32 \
    --verbose

### 3. Primary particle type classification
The model performs binary classification of the primary particle type, learning to distinguish between the two particle classes: signal (gamma-ray events) and background (proton events), based on the information contained in the camera image.
The input directory containing the signal events can be specified using *--TrainCTLearnModel.input_dir_signal* or the shorthand *--signal*.
One or more file patterns can be provided using *--TrainCTLearnModel.file_pattern_signal* or *--pattern-signal*.
The corresponding options for the background events are *--TrainCTLearnModel.input_dir_background* / *--background* and *--TrainCTLearnModel.file_pattern_background* / *--pattern-background*.

It is recommended to approximately balance the training dataset before training.
The training log reports the number of signal and background events remaining after the selection cuts, which can be used to check the class balance.
If a significant imbalance is present, the input file patterns should be adjusted to include a more balanced selection of gamma-ray and proton events.
CTLearn includes a standard, fixed mechanism that computes class weights and incorporates them into the loss calculation to compensate for differences in the number of signal and background events.
However, class weighting is only effective up to a certain degree of class imbalance. It is therefore important to monitor the training carefully, particularly when the input samples are not well balanced.
The training logs and learning curves should be inspected throughout the training. In particular, the evolution of metrics such as loss, accuracy, and AUC on a per-epoch basis can help identify potential issues arising from class imbalance or other training problems.

In [ ]:
! mkdir ../my_outputs
! ctlearn-train-model \
    --signal {TRAIN_DIR} \
    --pattern-signal "gamma-diffuse_with_images_*.dl2.h5" \
    --background {TRAIN_DIR} \
    --pattern-background "proton_with_images_*.dl2.h5" \
    --output "../my_outputs/my_first_training_type" \
    --config "../configs/dl1dh_example_config.json" \
    --config "../configs/ctlearn_train_model_example_config.json" \
    --TrainCTLearnModel.n_epochs 5 \
    --reco type \
    --verbose

## Customizing the CTLearn model architecture
CTLearn provides a flexible configuration system that allows users to customize the neural network architecture.

Two main CNN-based architectures are currently implemented: **SingleCNN** and **ResNet**, which can bet set by *--TrainCTLearnModel.model_type*.

The **SingleCNN** component implements a plain convolutional neural network consisting of configurable convolutional layers.
It can be used to construct very simple and lightweight models, but it can also be configured to create deeper architectures, such as VGG-like networks.
This architecture represents the more legacy model structure in CTLearn and was the starting point for early proof-of-concept studies exploring the applicability of CNNs to the problem.
For applications closer to the camera hardware and its readout (for the trigger system), particularly where predictions need to be made at very high rates and with very low latency, computational efficiency becomes increasingly important.

The **ResNet** component provides the currently preferred architecture for constructing deeper CNN models.
Residual networks allow substantially deeper architectures while maintaining efficient training and inference, and they generally provide improved performance compared with the simpler plain CNN architecture.
The architecture, number of layers or blocks, number of filters, and other relevant parameters can therefore be configured directly through the CTLearn configuration system, allowing the model complexity to be adapted to the requirements of a particular application.
For ResNet architectures, CTLearn currently supports two types of residual blocks:
- Basic blocks, which use the simpler residual block structure and are suitable for relatively lightweight networks.
- Bottleneck blocks, which use a more compact sequence of convolutions to enable deeper networks while controlling the computational cost.

In [ ]:
! mkdir ../my_outputs
! ctlearn-train-model \
    --signal {TRAIN_DIR} \
    --pattern-signal "gamma-diffuse_with_images_*.dl2.h5" \
    --TrainCTLearnModel.model_type "SingleCNN" \
    --output "../my_outputs/my_very_own_plain_CNN_energy" \
    --config "../configs/dl1dh_example_config.json" \
    --config "../configs/ctlearn_train_model_example_config.json" \
    --config "../configs/ctlearn_models_example_config.json" \
    --reco energy \
    --verbose

## Using a pre-trained model and transfer learning
The LoadedModel component allows a previously trained model following the CTLearnModel structure to be loaded for either resuming training or transfer learning.
Transfer learning reuses the learned features of an existing model and adapts them to new, but related, data or tasks.
This can reduce training time compared with training a model from scratch.

Two parameters are particularly important:
 - *--LoadedModel.trainable_backbone* controls whether the pre-trained backbone is frozen (false) or trainable (true).
 - *--LoadedModel.overwrite_head* controls whether the structure of the original prediction head is reused (false) or replaced with a new MLP tailored to the new task or data (true).

In [ ]:
! mkdir ../my_outputs
! ctlearn-train-model \
    --signal {TRAIN_DIR} \
    --pattern-signal "gamma-diffuse_with_images_*.dl2.h5" \
    --TrainCTLearnModel.model_type "LoadedModel" \
    --LoadedModel.load_model_from "../my_outputs/my_very_own_plain_CNN_energy/ctlearn_model.keras" \
    --output "../my_outputs/my_very_own_plain_CNN_energy_reloaded" \
    --config "../configs/dl1dh_example_config.json" \
    --config "../configs/ctlearn_train_model_example_config.json" \
    --config "../configs/ctlearn_models_example_config.json" \
    --reco energy \
    --verbose

## Monitoring training with TensorBoard
TensorBoard can be used to monitor the training process live.
CTLearn writes training metrics to the output directory, which can then be visualized with TensorBoard.
Once started, TensorBoard provides an interactive web interface where metrics such as loss, accuracy, and AUC can be monitored over the course of training.
This is useful for checking the training progress and identifying issues such as overfitting or unstable training.
The command can also be run directly from a terminal or on a remote/cluster environment, where the TensorBoard web interface can be accessed through the appropriate port forwarding.

In [ ]:
#! tensorboard --logdir ../my_outputs/

## Basic stereoscopic CNN-based model training
CTLearn currently provides a basic approach for stereoscopic CNN-based model training.
The images from the different telescopes are concatenated channel-wise and then passed to a standard, monoscopic CNN backbone as above.
In this way, the network can process the combined input while retaining stereoscopic information.
An earlier version of CTLearn also included a CNN-RNN architecture, using recurrent neural networks to explicitly handle information from multiple telescopes dynamically. This model has not yet been migrated to the latest CTLearn version based on TensorFlow 2 and Keras 3, so this tutorial focuses on the channel-wise concatenation approach.

The main configuration parameters for stereoscopic training are:
 - *--DLImageReader.mode*: set to "stereo" to enable stereoscopic image reading
 - *--DLImageReader.min_telescopes*: defines the minimum number of telescopes that must participate in an event after the quality cuts.
 - *--TrainCTLearnModel.stack_telescope_images*: set to true to stack/concatenate the images from the telescopes.

Since the resulting input contains information from multiple telescopes and is therefore larger than the corresponding monoscopic input, it is recommended to reduce the batch size using *--TrainCTLearnModel.batch_size* to account for the increased memory consumption.

In [ ]:
! mkdir ../my_outputs
! ctlearn-train-model \
    --signal {TRAIN_DIR} \
    --pattern-signal "gamma-diffuse_with_images_*.dl2.h5" \
    --output "../my_outputs/my_first_stereo_training_energy" \
    --config "../configs/dl1dh_example_config.json" \
    --config "../configs/ctlearn_train_model_example_config.json" \
    --DLImageReader.mode stereo \
    --DLImageReader.min_telescopes 2 \
    --TrainCTLearnModel.stack_telescope_images True \
    --TrainCTLearnModel.batch_size 16 \
    --reco energy \
    --verbose